In [ ]:
import torchvision
from torchvision import models

In [ ]:
models.list_models()


In [ ]:
alexnet = models.AlexNet()

In [ ]:
vit = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)

In [ ]:
vit


In [ ]:
from torchvision import transforms
preprocess = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )])


In [ ]:
from PIL import Image
img = Image.open("data/bobby.jpg")

In [ ]:
img

In [ ]:
img_t = preprocess(img)
img_t.shape

In [ ]:
import torch
batch_t = torch.unsqueeze(img_t, 0)

In [ ]:
vit.eval()

In [ ]:
out = vit(batch_t)
out

In [ ]:
with open('data/imagenet_classes.txt') as f:
    labels = [line.strip() for line in f.readlines()]


In [ ]:
_, index = torch.max(out, 1)

In [ ]:
percentage = torch.nn.functional.softmax(out, dim=1)[0] * 100
labels[index.item()], percentage[index.item()].item()

In [ ]:
_, indices = torch.sort(out, descending=True)
[(labels[idx], percentage[idx].item()) for idx in indices[0][:5]]

In [ ]:
from diffusers import StableDiffusionInpaintPipeline
import torch
import PIL.Image as Image

device ="cuda"
pipe = StableDiffusionInpaintPipeline.from_pretrained(
    'sd2-community/stable-diffusion-2-inpainting',
    torch_dtype = torch.float16
).to(device)

In [ ]:
pipe

In [ ]:
img = Image.open("data/horse.jpg")
img


In [ ]:
mask_img = Image.open("data/horse_mask.jpg")

prompt = ("a zebra replacing the original horse, same pose, same lighting, background unchanged")
negative = "distorted background, blurry, text, watermark"

out = pipe(
    prompt=prompt,
    image=img,
    mask_image=mask_img,
    negative_prompt=negative,
    guidance_scale=7.5,
    strength=.8,
    generator=torch.Generator(device).manual_seed(42)
)

In [ ]:
out.images[0]

In [ ]:
from transformers import pipeline
generator = pipeline('text-generation', model = 'gpt2')
generator('hello, my name is ', max_new_tokens=5)

In [ ]:
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-large")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-large")

In [ ]:
def annotate_image(image: Image) -> None:
    display(image)
    inputs = processor(image, return_tensors="pt")
    out = model.generate(**inputs)
    print(processor.decode(out[0], skip_special_tokens=True))
    print(processor.decode(out))


In [ ]:
annotate_image(img)

In [ ]:
annotate_image(out.images[0])

In [ ]:
img = Image.open('data/bobby.jpg')
annotate_image(img)

In [ ]:
mask_img = Image.open("data/bobby.jpg")

prompt = ("a zebra replacing the original horse, same pose, same lighting, background unchanged")
negative = "distorted background, blurry, text, watermark"

out = pipe(
    prompt=prompt,
    image=img,
    mask_image=mask_img,
    negative_prompt=negative,
    guidance_scale=7.5,
    strength=.8,
    generator=torch.Generator(device).manual_seed(42)
)

out.images[0]